# Imports

In [ ]:
import pandas as pd

# Load

In [ ]:
df_enade = pd.read_parquet("../2_enade_staging/enade18-23_mapped.parquet")

df_municipios = pd.read_parquet("../2_enade_staging/municipios.parquet")

df_cursos = pd.read_parquet("../2_enade_staging/cursos.parquet")

Merge das tabelas

In [10]:
df_desnormalizado = df_enade.merge(df_municipios, on="id_municipio", how="left").merge(df_cursos, on="id_curso", how="left")

# Criando indices únicos para agrupamentos

In [14]:
copia = df_desnormalizado.copy()

indice de participação

In [13]:
df_desnormalizado.insert(0, "id_participacao", range(1, len(df_desnormalizado) + 1))

Colunas dos agrupamentos

In [15]:
cols_desempenho = ["nota_geral", "nota_formacao_geral", "nota_componentes_especifico"]

cols_avaliacao = ["avaliacao_dificuldade_fg", "avaliacao_dificuldade_ce", "avaliacao_equipamentos", "avaliacao_ambiente"]

cols_perfil_estudante = [
    "sexo", "idade", "cor_raca", "ano_fim_ensino_medio", "ano_ingresso_graduacao", "tipo_escola_ensino_medio", "primeira_geracao", "escolaridade_pai", "escolaridade_mae", "motivacao_curso", "renda_familiar", "horas_trabalho", "cotas"]

cols_oferta = ['id_curso', 'id_municipio']

Criando Dimensões

In [16]:
dim_desempenho = (copia[cols_desempenho].drop_duplicates().reset_index(drop=True))

dim_avaliacao = (copia[cols_avaliacao].drop_duplicates().reset_index(drop=True))

dim_perfil_estudante = (copia[cols_perfil_estudante].drop_duplicates().reset_index(drop=True))

dim_oferta = (copia[cols_oferta].drop_duplicates().reset_index(drop=True))

Criando índices únicos para dimensões

In [17]:
dim_desempenho["id_desempenho"] = dim_desempenho.index + 1
dim_avaliacao["id_avaliacao"] = dim_avaliacao.index + 1
dim_perfil_estudante["id_perfil_estudante"] = dim_perfil_estudante.index + 1
dim_oferta["id_oferta"] = dim_oferta.index + 1


Concatenando os índices as dimensões

In [18]:
dim_desempenho = dim_desempenho[["id_desempenho"] + cols_desempenho]
dim_avaliacao = dim_avaliacao[["id_avaliacao"] + cols_avaliacao]
dim_perfil_estudante = dim_perfil_estudante[["id_perfil_estudante"] + cols_perfil_estudante]
dim_oferta = dim_oferta[["id_oferta"] + cols_oferta]

Merge dos indíces de dimensões a tabela original

In [19]:
copia = copia.merge(dim_desempenho, on=cols_desempenho, how="left").merge(dim_avaliacao, on=cols_avaliacao, how="left").merge(dim_perfil_estudante, on=cols_perfil_estudante, how="left").merge(dim_oferta, on=cols_oferta, how="left")

Reordenando Colunas

In [20]:
ordem_certa = [
    'id_participacao',
    'id_oferta',
    'id_curso', 
    'nome_curso',
    'categoria_administrativa',
    'modalidade_graduacao',
    'turno_graduacao',
    'ano_enade',
    'id_municipio',
    'nome_municipio', 
    'uf',
    'regiao',
    'id_perfil_estudante', 
    'sexo',
    'idade', 
    'cor_raca',
    'ano_fim_ensino_medio', 
    'ano_ingresso_graduacao',
    'tipo_escola_ensino_medio', 
    'primeira_geracao', 
    'escolaridade_pai',
    'escolaridade_mae', 
    'motivacao_curso', 
    'renda_familiar',
    'horas_trabalho', 
    'cotas',
    'id_desempenho', 
    'nota_geral', 
    'nota_formacao_geral',
    'nota_componentes_especifico',
    'id_avaliacao', 
    'avaliacao_dificuldade_fg',
    'avaliacao_dificuldade_ce', 
    'avaliacao_equipamentos',
    'avaliacao_ambiente'
    ]

copia = copia[ordem_certa]

# Salvando Tabela única com todas colunas prontas "Enade_Desnormalizado"

Salva em formato .csv (Arquivo Pesado)

In [ ]:
copia.to_csv("../3_enade_ready/enade_desnormalizado.csv", index=False, encoding="utf-8-sig")

Salva em formato .parquet (Arquivo Leve)

In [ ]:
copia.to_parquet("../3_enade_ready/enade_desnormalizado.parquet", index=False)

Salvando amostra para conferir 

In [ ]:
head = copia.sample(frac=0.1).reset_index(drop=True)
head.head(5).to_csv("../3_enade_ready/head.csv", index=False, encoding="utf-8-sig")